# Sanskrit OCR Pipeline - Complete Workflow

This notebook provides a complete pipeline for Sanskrit OCR processing, similar to Google Colab workflow.

## Workflow Overview:
1. **Install Dependencies** - Install all required packages
2. **Import Libraries** - Import necessary Python libraries  
3. **Load Dataset** - Load images from folder and create dataset
4. **Data Quality Check** - Check for null values and corrupted files
5. **Image Validation** - Identify empty/invalid images
6. **Preprocessing** - Clean and prepare images for OCR
7. **Line Segmentation** - Extract text lines from images
8. **Text Recognition** - Convert images to text using AI models
9. **Post-processing** - Clean and format extracted text
10. **Results Analysis** - Analyze and save results

## 1. Install Dependencies
First, let's install all the required packages for our Sanskrit OCR pipeline.

In [ ]:
# Install all required dependencies
!pip install opencv-python numpy pillow pandas matplotlib seaborn
!pip install transformers torch torchvision 
!pip install pytesseract scikit-learn plotly
!pip install requests beautifulsoup4
!pip install tqdm jupyter-widgets

print("✅ All dependencies installed successfully!")

## 2. Import Required Libraries
Now let's import all the necessary libraries for our Sanskrit OCR pipeline.

In [ ]:
# Core libraries for data handling and processing
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Image processing libraries
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

# Machine Learning and OCR libraries
try:
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    import torch
    TROCR_AVAILABLE = True
    print("✅ TrOCR model available")
except ImportError:
    TROCR_AVAILABLE = False
    print("⚠️  TrOCR not available, will use Tesseract as fallback")

try:
    import pytesseract
    TESSERACT_AVAILABLE = True
    print("✅ Tesseract OCR available")
except ImportError:
    TESSERACT_AVAILABLE = False
    print("❌ Tesseract OCR not available")

# Utility libraries
from tqdm import tqdm
from sklearn.metrics import accuracy_score
import re
from typing import List, Tuple, Dict
import glob
from pathlib import Path

print("📚 All libraries imported successfully!")
print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 3. Load Dataset from Folder
Let's load all Sanskrit manuscript images from the specified folder and create a structured dataset.

In [ ]:
# Configuration - CHANGE THIS PATH TO YOUR DATASET FOLDER
DATASET_FOLDER = r"d:\Projects\Sanskrit-OCR\maitrayani_dataset\line_images"
OUTPUT_FOLDER = r"d:\Projects\Sanskrit-OCR\ocr_results"

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

def load_image_dataset(folder_path: str) -> pd.DataFrame:
    """Load all images from folder and create a structured dataset"""
    
    # Supported image formats
    image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.tiff', '*.bmp', '*.PNG', '*.JPG', '*.JPEG']
    
    # Find all image files
    image_files = []
    for extension in image_extensions:
        image_files.extend(glob.glob(os.path.join(folder_path, extension)))
    
    # Create dataset
    dataset_info = []
    
    print(f"🔍 Scanning folder: {folder_path}")
    print(f"📁 Found {len(image_files)} image files")
    
    for image_path in tqdm(image_files, desc="Loading images"):
        file_info = {
            'image_path': image_path,
            'filename': os.path.basename(image_path),
            'file_size_kb': os.path.getsize(image_path) / 1024,
            'file_extension': os.path.splitext(image_path)[1].lower(),
            'exists': os.path.exists(image_path),
            'readable': None,
            'width': None,
            'height': None,
            'channels': None,
            'is_valid': None
        }
        dataset_info.append(file_info)
    
    # Convert to DataFrame
    df = pd.DataFrame(dataset_info)
    return df

# Load the dataset
print("📊 Creating dataset from images...")
dataset_df = load_image_dataset(DATASET_FOLDER)

print(f"\n📈 Dataset Summary:")
print(f"Total images found: {len(dataset_df)}")
print(f"Average file size: {dataset_df['file_size_kb'].mean():.2f} KB")
print(f"File extensions: {dataset_df['file_extension'].value_counts().to_dict()}")

# Display first few rows
print(f"\n📋 First 5 entries:")
print(dataset_df[['filename', 'file_size_kb', 'file_extension']].head())

## 4. Check for Null Values and Missing Data
Let's identify any missing files, null values, or corrupted file paths in our dataset.

In [ ]:
def check_dataset_quality(df: pd.DataFrame) -> Dict:
    """Check for null values, missing files, and data quality issues"""
    
    quality_report = {
        'total_images': len(df),
        'null_values': {},
        'missing_files': 0,
        'file_size_issues': 0,
        'data_quality_score': 0
    }
    
    # Check for null values in each column
    print("🔍 Checking for null values...")
    for column in df.columns:
        null_count = df[column].isnull().sum()
        if null_count > 0:
            quality_report['null_values'][column] = null_count
            print(f"⚠️  {column}: {null_count} null values")
    
    # Check for missing files
    print("\n📂 Checking file existence...")
    missing_files = df[~df['exists']]
    quality_report['missing_files'] = len(missing_files)
    
    if len(missing_files) > 0:
        print(f"❌ {len(missing_files)} files are missing:")
        for _, row in missing_files.head().iterrows():
            print(f"   - {row['filename']}")
    else:
        print("✅ All files exist")
    
    # Check for suspiciously small files (likely corrupted)
    print("\n📏 Checking file sizes...")
    small_files = df[df['file_size_kb'] < 1]  # Files smaller than 1KB
    quality_report['file_size_issues'] = len(small_files)
    
    if len(small_files) > 0:
        print(f"⚠️  {len(small_files)} files are suspiciously small (< 1KB):")
        for _, row in small_files.head().iterrows():
            print(f"   - {row['filename']}: {row['file_size_kb']:.2f} KB")
    else:
        print("✅ All files have reasonable sizes")
    
    # Calculate overall quality score
    total_issues = (len(quality_report['null_values']) + 
                   quality_report['missing_files'] + 
                   quality_report['file_size_issues'])
    
    quality_report['data_quality_score'] = max(0, 100 - (total_issues / len(df) * 100))
    
    return quality_report

# Run quality check
quality_report = check_dataset_quality(dataset_df)

print(f"\n📊 Data Quality Report:")
print(f"Total images: {quality_report['total_images']}")
print(f"Missing files: {quality_report['missing_files']}")
print(f"File size issues: {quality_report['file_size_issues']}")
print(f"Data quality score: {quality_report['data_quality_score']:.1f}/100")

# Show basic statistics
print(f"\n📈 File Size Statistics:")
print(dataset_df['file_size_kb'].describe())

## 5. Check for Empty and Corrupted Images
Now let's examine each image to identify empty, corrupted, or unreadable files.

In [ ]:
def validate_images(df: pd.DataFrame) -> pd.DataFrame:
    """Check each image for corruption, emptiness, and extract basic properties"""
    
    print("🖼️  Validating images...")
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Validating images"):
        image_path = row['image_path']
        
        try:
            # Try to load image with OpenCV
            img = cv2.imread(image_path)
            
            if img is None:
                # Try with PIL as backup
                try:
                    pil_img = Image.open(image_path)
                    img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
                except Exception:
                    img = None
            
            if img is not None:
                # Image is readable
                height, width, channels = img.shape
                df.at[idx, 'readable'] = True
                df.at[idx, 'width'] = width
                df.at[idx, 'height'] = height
                df.at[idx, 'channels'] = channels
                df.at[idx, 'is_valid'] = True
                
                # Check if image is too small (likely empty or corrupted)
                if width < 50 or height < 50:
                    df.at[idx, 'is_valid'] = False
                    
            else:
                # Image is not readable
                df.at[idx, 'readable'] = False
                df.at[idx, 'is_valid'] = False
                
        except Exception as e:
            df.at[idx, 'readable'] = False
            df.at[idx, 'is_valid'] = False
    
    return df

# Validate all images
dataset_df = validate_images(dataset_df)

# Analysis of image validation results
valid_images = dataset_df[dataset_df['is_valid'] == True]
invalid_images = dataset_df[dataset_df['is_valid'] == False]
unreadable_images = dataset_df[dataset_df['readable'] == False]

print(f"\n📊 Image Validation Results:")
print(f"✅ Valid images: {len(valid_images)}")
print(f"❌ Invalid images: {len(invalid_images)}")
print(f"🚫 Unreadable images: {len(unreadable_images)}")

if len(invalid_images) > 0:
    print(f"\n⚠️  Invalid/Problematic Images:")
    for _, row in invalid_images.head(10).iterrows():
        status = "Unreadable" if not row['readable'] else "Too small"
        print(f"   - {row['filename']}: {status}")

# Show image dimension statistics for valid images
if len(valid_images) > 0:
    print(f"\n📏 Image Dimension Statistics (Valid Images Only):")
    print(f"Width - Min: {valid_images['width'].min()}, Max: {valid_images['width'].max()}, Mean: {valid_images['width'].mean():.1f}")
    print(f"Height - Min: {valid_images['height'].min()}, Max: {valid_images['height'].max()}, Mean: {valid_images['height'].mean():.1f}")

# Filter dataset to keep only valid images
print(f"\n🔄 Filtering dataset to keep only valid images...")
clean_dataset_df = valid_images.copy()
print(f"📊 Clean dataset size: {len(clean_dataset_df)} images")

## 6. Visualize Sample Images
Let's look at some sample images from our dataset to understand what we're working with.

In [ ]:
def display_sample_images(df: pd.DataFrame, num_samples: int = 6):
    """Display sample images from the dataset"""
    
    sample_images = df.sample(min(num_samples, len(df)))
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()
    
    for idx, (_, row) in enumerate(sample_images.iterrows()):
        if idx >= num_samples:
            break
            
        try:
            img = cv2.imread(row['image_path'])
            if img is not None:
                # Convert BGR to RGB for matplotlib
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                
                axes[idx].imshow(img_rgb)
                axes[idx].set_title(f"{row['filename']}\n{int(row['width'])}x{int(row['height'])}")
                axes[idx].axis('off')
            else:
                axes[idx].text(0.5, 0.5, 'Image\nNot Readable', 
                              ha='center', va='center', transform=axes[idx].transAxes)
                axes[idx].set_title(f"{row['filename']}")
        except Exception as e:
            axes[idx].text(0.5, 0.5, f'Error:\n{str(e)[:50]}', 
                          ha='center', va='center', transform=axes[idx].transAxes)
            axes[idx].set_title(f"{row['filename']}")
    
    # Hide unused subplots
    for idx in range(len(sample_images), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.suptitle('Sample Sanskrit Manuscript Images', y=1.02, fontsize=16)
    plt.show()

# Display sample images
if len(clean_dataset_df) > 0:
    print("🖼️  Displaying sample images from the dataset...")
    display_sample_images(clean_dataset_df, num_samples=6)
else:
    print("❌ No valid images found to display")

## 7. Image Preprocessing Pipeline
Now let's preprocess the images to make them optimal for OCR processing.

In [ ]:
class ImagePreprocessor:
    """Complete image preprocessing pipeline for Sanskrit OCR"""
    
    def __init__(self):
        self.preprocessing_stats = {
            'total_processed': 0,
            'successful_preprocessing': 0,
            'failed_preprocessing': 0
        }
    
    def preprocess_image(self, image_path: str, show_steps: bool = False) -> dict:
        """Complete preprocessing pipeline for a single image"""
        
        try:
            # Load original image
            original = cv2.imread(image_path)
            if original is None:
                return {'success': False, 'error': 'Could not load image'}
            
            # Step 1: Convert to grayscale
            gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
            
            # Step 2: Noise reduction (for old manuscript pages)
            denoised = cv2.fastNlMeansDenoising(gray)
            
            # Step 3: Binarization (black text, white background)
            binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
            
            # Step 4: Morphological operations to clean up text
            kernel = np.ones((1, 1), np.uint8)
            cleaned = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
            
            # Step 5: Resize if image is too large (optional)
            height, width = cleaned.shape
            if width > 2000:  # Resize if width > 2000px
                scale_factor = 2000 / width
                new_width = 2000
                new_height = int(height * scale_factor)
                cleaned = cv2.resize(cleaned, (new_width, new_height), interpolation=cv2.INTER_AREA)
            
            # Show processing steps if requested
            if show_steps:
                self._show_preprocessing_steps(original, gray, denoised, binary, cleaned)
            
            result = {
                'success': True,
                'original_shape': original.shape,
                'processed_shape': cleaned.shape,
                'original_image': original,
                'processed_image': cleaned,
                'preprocessing_steps': {
                    'grayscale': gray,
                    'denoised': denoised,
                    'binary': binary,
                    'cleaned': cleaned
                }
            }
            
            self.preprocessing_stats['successful_preprocessing'] += 1
            return result
            
        except Exception as e:
            self.preprocessing_stats['failed_preprocessing'] += 1
            return {'success': False, 'error': str(e)}
        
        finally:
            self.preprocessing_stats['total_processed'] += 1
    
    def _show_preprocessing_steps(self, original, gray, denoised, binary, cleaned):
        """Visualize preprocessing steps"""
        
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        
        # Original
        axes[0, 0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
        axes[0, 0].set_title('1. Original Image')
        axes[0, 0].axis('off')
        
        # Grayscale
        axes[0, 1].imshow(gray, cmap='gray')
        axes[0, 1].set_title('2. Grayscale')
        axes[0, 1].axis('off')
        
        # Denoised
        axes[0, 2].imshow(denoised, cmap='gray')
        axes[0, 2].set_title('3. Denoised')
        axes[0, 2].axis('off')
        
        # Binary
        axes[1, 0].imshow(binary, cmap='gray')
        axes[1, 0].set_title('4. Binary (Otsu)')
        axes[1, 0].axis('off')
        
        # Cleaned
        axes[1, 1].imshow(cleaned, cmap='gray')
        axes[1, 1].set_title('5. Final Cleaned')
        axes[1, 1].axis('off')
        
        # Hide the last subplot
        axes[1, 2].axis('off')
        
        plt.tight_layout()
        plt.show()
    
    def get_preprocessing_stats(self):
        """Get preprocessing statistics"""
        return self.preprocessing_stats

# Initialize preprocessor
preprocessor = ImagePreprocessor()

# Test preprocessing on a sample image
if len(clean_dataset_df) > 0:
    sample_image_path = clean_dataset_df.iloc[0]['image_path']
    print(f"🔧 Testing preprocessing on: {os.path.basename(sample_image_path)}")
    
    result = preprocessor.preprocess_image(sample_image_path, show_steps=True)
    
    if result['success']:
        print("✅ Preprocessing successful!")
        print(f"Original shape: {result['original_shape']}")
        print(f"Processed shape: {result['processed_shape']}")
    else:
        print(f"❌ Preprocessing failed: {result['error']}")
else:
    print("❌ No valid images available for preprocessing test")

## 8. Line Segmentation 
Extract individual text lines from preprocessed images for better OCR accuracy.

In [ ]:
class LineSegmenter:
    """Extract text lines from preprocessed images"""
    
    def __init__(self):
        self.segmentation_stats = {
            'images_processed': 0,
            'total_lines_found': 0,
            'average_lines_per_image': 0
        }
    
    def segment_lines(self, binary_image: np.ndarray, min_line_height: int = 20, show_process: bool = False) -> List[np.ndarray]:
        """Segment text lines using horizontal projection"""
        
        # Calculate horizontal projection (sum of black pixels in each row)
        h_projection = np.sum(binary_image == 0, axis=1)
        
        # Find line boundaries
        lines = []
        in_line = False
        start_y = 0
        
        for i, pixel_count in enumerate(h_projection):
            if pixel_count > 10 and not in_line:  # Start of a line
                start_y = i
                in_line = True
            elif pixel_count <= 10 and in_line:  # End of a line
                if i - start_y > min_line_height:  # Filter out noise
                    line_image = binary_image[start_y:i, :]
                    lines.append(line_image)
                in_line = False
        
        # Handle case where image ends while in a line
        if in_line and binary_image.shape[0] - start_y > min_line_height:
            line_image = binary_image[start_y:, :]
            lines.append(line_image)
        
        # Show segmentation process if requested
        if show_process:
            self._visualize_segmentation(binary_image, h_projection, lines)
        
        # Update statistics
        self.segmentation_stats['images_processed'] += 1
        self.segmentation_stats['total_lines_found'] += len(lines)
        self.segmentation_stats['average_lines_per_image'] = (
            self.segmentation_stats['total_lines_found'] / 
            self.segmentation_stats['images_processed']
        )
        
        return lines
    
    def _visualize_segmentation(self, binary_image: np.ndarray, h_projection: np.ndarray, lines: List[np.ndarray]):
        """Visualize the line segmentation process"""
        
        fig, axes = plt.subplots(1, 3, figsize=(20, 8))
        
        # Original binary image
        axes[0].imshow(binary_image, cmap='gray')
        axes[0].set_title('Original Binary Image')
        axes[0].axis('off')
        
        # Horizontal projection
        axes[1].plot(h_projection, range(len(h_projection)))
        axes[1].invert_yaxis()
        axes[1].set_title('Horizontal Projection')
        axes[1].set_xlabel('Pixel Count')
        axes[1].set_ylabel('Row')
        axes[1].grid(True, alpha=0.3)
        
        # Segmented lines
        if lines:
            # Show first few lines as examples
            num_lines_to_show = min(len(lines), 5)
            combined_lines = np.vstack(lines[:num_lines_to_show])
            axes[2].imshow(combined_lines, cmap='gray')
            axes[2].set_title(f'First {num_lines_to_show} Segmented Lines')
        else:
            axes[2].text(0.5, 0.5, 'No lines found', ha='center', va='center', transform=axes[2].transAxes)
            axes[2].set_title('No Lines Found')
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
    
    def get_segmentation_stats(self):
        """Get segmentation statistics"""
        return self.segmentation_stats

# Initialize line segmenter
line_segmenter = LineSegmenter()

# Test line segmentation on preprocessed sample
if len(clean_dataset_df) > 0:
    sample_image_path = clean_dataset_df.iloc[0]['image_path']
    print(f"🔪 Testing line segmentation on: {os.path.basename(sample_image_path)}")
    
    # First preprocess the image
    preprocess_result = preprocessor.preprocess_image(sample_image_path)
    
    if preprocess_result['success']:
        # Segment lines
        lines = line_segmenter.segment_lines(
            preprocess_result['processed_image'], 
            show_process=True
        )
        
        print(f"✅ Line segmentation successful!")
        print(f"Found {len(lines)} text lines")
        
        # Display individual lines
        if lines:
            fig, axes = plt.subplots(len(lines), 1, figsize=(15, 2*len(lines)))
            if len(lines) == 1:
                axes = [axes]
            
            for i, line in enumerate(lines):
                axes[i].imshow(line, cmap='gray')
                axes[i].set_title(f'Line {i+1}')
                axes[i].axis('off')
            
            plt.tight_layout()
            plt.suptitle('Individual Text Lines', y=1.02)
            plt.show()
    else:
        print(f"❌ Could not preprocess image: {preprocess_result['error']}")
else:
    print("❌ No valid images available for line segmentation test")

## 9. Text Recognition using AI Models
Now let's use advanced AI models to convert image text to actual Sanskrit text.

In [ ]:
class TextRecognizer:
    """Advanced text recognition using multiple OCR engines"""
    
    def __init__(self):
        self.recognition_stats = {
            'lines_processed': 0,
            'successful_recognitions': 0,
            'failed_recognitions': 0,
            'trocr_used': 0,
            'tesseract_used': 0
        }
        
        # Initialize TrOCR if available
        if TROCR_AVAILABLE:
            try:
                print("🚀 Loading TrOCR model...")
                self.processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
                self.model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed")
                self.trocr_ready = True
                print("✅ TrOCR model loaded successfully!")
            except Exception as e:
                print(f"⚠️  Could not load TrOCR: {e}")
                self.trocr_ready = False
        else:
            self.trocr_ready = False
    
    def recognize_text_trocr(self, line_image: np.ndarray) -> str:
        """Recognize text using Microsoft TrOCR"""
        try:
            if not self.trocr_ready:
                return ""
            
            # Convert numpy array to PIL Image
            pil_image = Image.fromarray(line_image)
            
            # Process with TrOCR
            pixel_values = self.processor(pil_image, return_tensors="pt").pixel_values
            generated_ids = self.model.generate(pixel_values, max_length=100)
            generated_text = self.processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
            
            self.recognition_stats['trocr_used'] += 1
            return generated_text
            
        except Exception as e:
            print(f"TrOCR error: {e}")
            return ""
    
    def recognize_text_tesseract(self, line_image: np.ndarray) -> str:
        """Recognize text using Tesseract OCR"""
        try:
            if not TESSERACT_AVAILABLE:
                return ""
            
            # Convert numpy array to PIL Image
            pil_image = Image.fromarray(line_image)
            
            # Use Tesseract with Sanskrit configuration
            custom_config = r'-l san --oem 3 --psm 6'
            text = pytesseract.image_to_string(pil_image, config=custom_config)
            
            self.recognition_stats['tesseract_used'] += 1
            return text.strip()
            
        except Exception as e:
            print(f"Tesseract error: {e}")
            return ""
    
    def recognize_line(self, line_image: np.ndarray, prefer_trocr: bool = True) -> dict:
        """Recognize text from a single line image"""
        
        result = {
            'trocr_text': '',
            'tesseract_text': '',
            'final_text': '',
            'method_used': '',
            'success': False
        }
        
        try:
            # Try TrOCR first if preferred and available
            if prefer_trocr and self.trocr_ready:
                trocr_text = self.recognize_text_trocr(line_image)
                result['trocr_text'] = trocr_text
                
                if trocr_text:
                    result['final_text'] = trocr_text
                    result['method_used'] = 'TrOCR'
                    result['success'] = True
            
            # Try Tesseract if TrOCR failed or not preferred
            if not result['success'] and TESSERACT_AVAILABLE:
                tesseract_text = self.recognize_text_tesseract(line_image)
                result['tesseract_text'] = tesseract_text
                
                if tesseract_text:
                    result['final_text'] = tesseract_text
                    result['method_used'] = 'Tesseract'
                    result['success'] = True
            
            # Try both and combine if nothing worked so far
            if not result['success']:
                if not result['trocr_text'] and self.trocr_ready:
                    result['trocr_text'] = self.recognize_text_trocr(line_image)
                if not result['tesseract_text'] and TESSERACT_AVAILABLE:
                    result['tesseract_text'] = self.recognize_text_tesseract(line_image)
                
                # Use the longer result as final
                if len(result['trocr_text']) > len(result['tesseract_text']):
                    result['final_text'] = result['trocr_text']
                    result['method_used'] = 'TrOCR'
                elif result['tesseract_text']:
                    result['final_text'] = result['tesseract_text']
                    result['method_used'] = 'Tesseract'
                
                result['success'] = bool(result['final_text'])
            
            # Update stats
            if result['success']:
                self.recognition_stats['successful_recognitions'] += 1
            else:
                self.recognition_stats['failed_recognitions'] += 1
                
        except Exception as e:
            result['error'] = str(e)
            self.recognition_stats['failed_recognitions'] += 1
        
        finally:
            self.recognition_stats['lines_processed'] += 1
        
        return result
    
    def get_recognition_stats(self):
        """Get recognition statistics"""
        return self.recognition_stats

# Initialize text recognizer
text_recognizer = TextRecognizer()

# Test text recognition on segmented lines
if len(clean_dataset_df) > 0:
    sample_image_path = clean_dataset_df.iloc[0]['image_path']
    print(f"🔤 Testing text recognition on: {os.path.basename(sample_image_path)}")
    
    # Preprocess and segment the image
    preprocess_result = preprocessor.preprocess_image(sample_image_path)
    
    if preprocess_result['success']:
        lines = line_segmenter.segment_lines(preprocess_result['processed_image'])
        
        if lines:
            print(f"🔍 Processing {len(lines)} lines for text recognition...")
            
            recognition_results = []
            for i, line in enumerate(lines[:3]):  # Test first 3 lines
                print(f"Processing line {i+1}...")
                result = text_recognizer.recognize_line(line)
                recognition_results.append(result)
                
                if result['success']:
                    print(f"✅ Line {i+1}: {result['final_text'][:100]}... ({result['method_used']})")
                else:
                    print(f"❌ Line {i+1}: Recognition failed")
            
            # Show statistics
            stats = text_recognizer.get_recognition_stats()
            print(f"\n📊 Recognition Statistics:")
            print(f"Lines processed: {stats['lines_processed']}")
            print(f"Successful: {stats['successful_recognitions']}")
            print(f"Failed: {stats['failed_recognitions']}")
            print(f"TrOCR used: {stats['trocr_used']}")
            print(f"Tesseract used: {stats['tesseract_used']}")
            
        else:
            print("❌ No lines found for text recognition")
    else:
        print(f"❌ Could not preprocess image: {preprocess_result['error']}")
else:
    print("❌ No valid images available for text recognition test")

## 10. Complete OCR Pipeline
Now let's combine all the components into a complete Sanskrit OCR pipeline.

In [ ]:
class SanskritOCRPipeline:
    """Complete Sanskrit OCR Pipeline"""
    
    def __init__(self):
        self.preprocessor = ImagePreprocessor()
        self.line_segmenter = LineSegmenter()
        self.text_recognizer = TextRecognizer()
        
        self.pipeline_stats = {
            'total_images': 0,
            'successful_images': 0,
            'failed_images': 0,
            'total_lines_processed': 0,
            'total_text_extracted': 0
        }
    
    def process_single_image(self, image_path: str) -> dict:
        """Process a single image through the complete pipeline"""
        
        result = {
            'image_path': image_path,
            'filename': os.path.basename(image_path),
            'success': False,
            'error': None,
            'preprocessing_result': None,
            'lines_found': 0,
            'lines_with_text': 0,
            'extracted_lines': [],
            'full_text': '',
            'processing_time': 0,
            'timestamp': datetime.now().isoformat()
        }
        
        start_time = datetime.now()
        
        try:
            # Step 1: Preprocessing
            preprocess_result = self.preprocessor.preprocess_image(image_path)
            result['preprocessing_result'] = preprocess_result['success']
            
            if not preprocess_result['success']:
                result['error'] = f"Preprocessing failed: {preprocess_result['error']}"
                return result
            
            # Step 2: Line Segmentation
            lines = self.line_segmenter.segment_lines(preprocess_result['processed_image'])
            result['lines_found'] = len(lines)
            
            if not lines:
                result['error'] = "No text lines found in image"
                return result
            
            # Step 3: Text Recognition for each line
            extracted_lines = []
            successful_lines = 0
            
            for i, line in enumerate(lines):
                line_result = self.text_recognizer.recognize_line(line)
                
                line_info = {
                    'line_number': i + 1,
                    'text': line_result['final_text'],
                    'method_used': line_result['method_used'],
                    'success': line_result['success']
                }
                
                extracted_lines.append(line_info)
                
                if line_result['success']:
                    successful_lines += 1
            
            result['extracted_lines'] = extracted_lines
            result['lines_with_text'] = successful_lines
            result['full_text'] = ' '.join([line['text'] for line in extracted_lines if line['success']])
            result['success'] = successful_lines > 0
            
            # Update pipeline stats
            self.pipeline_stats['successful_images'] += 1
            self.pipeline_stats['total_lines_processed'] += len(lines)
            self.pipeline_stats['total_text_extracted'] += len(result['full_text'])
            
        except Exception as e:
            result['error'] = str(e)
            self.pipeline_stats['failed_images'] += 1
        
        finally:
            self.pipeline_stats['total_images'] += 1
            result['processing_time'] = (datetime.now() - start_time).total_seconds()
        
        return result
    
    def process_dataset(self, df: pd.DataFrame, max_images: int = None) -> List[dict]:
        """Process entire dataset through the OCR pipeline"""
        
        if max_images:
            df_to_process = df.head(max_images)
        else:
            df_to_process = df
        
        print(f"🚀 Starting OCR processing for {len(df_to_process)} images...")
        
        all_results = []
        
        for idx, row in tqdm(df_to_process.iterrows(), total=len(df_to_process), desc="Processing images"):
            image_path = row['image_path']
            result = self.process_single_image(image_path)
            all_results.append(result)
            
            # Show progress every 10 images
            if (idx + 1) % 10 == 0:
                success_rate = (self.pipeline_stats['successful_images'] / self.pipeline_stats['total_images']) * 100
                print(f"Processed {idx + 1}/{len(df_to_process)} images. Success rate: {success_rate:.1f}%")
        
        return all_results
    
    def get_pipeline_stats(self):
        """Get complete pipeline statistics"""
        return self.pipeline_stats

# Initialize the complete OCR pipeline
ocr_pipeline = SanskritOCRPipeline()

# Process a small batch first (for testing)
if len(clean_dataset_df) > 0:
    # Process first 5 images as a test
    test_images = min(5, len(clean_dataset_df))
    print(f"🧪 Testing OCR pipeline on {test_images} images...")
    
    test_results = ocr_pipeline.process_dataset(clean_dataset_df, max_images=test_images)
    
    # Show results
    print(f"\n📊 Test Results Summary:")
    successful = sum(1 for r in test_results if r['success'])
    print(f"Successfully processed: {successful}/{test_images} images")
    
    # Show sample results
    for i, result in enumerate(test_results[:3]):
        print(f"\n📄 Image {i+1}: {result['filename']}")
        if result['success']:
            print(f"   Lines found: {result['lines_found']}")
            print(f"   Lines with text: {result['lines_with_text']}")
            print(f"   Sample text: {result['full_text'][:100]}...")
            print(f"   Processing time: {result['processing_time']:.2f}s")
        else:
            print(f"   ❌ Failed: {result['error']}")
    
    # Show pipeline statistics
    stats = ocr_pipeline.get_pipeline_stats()
    print(f"\n📈 Pipeline Statistics:")
    print(f"Total images processed: {stats['total_images']}")
    print(f"Successful: {stats['successful_images']}")
    print(f"Failed: {stats['failed_images']}")
    print(f"Total lines processed: {stats['total_lines_processed']}")
    print(f"Total text extracted: {stats['total_text_extracted']} characters")
else:
    print("❌ No valid images available for OCR processing")

## 11. Save Results and Analysis
Save the OCR results and perform comprehensive analysis.

In [ ]:
def save_and_analyze_results(results: List[dict], output_folder: str):
    """Save OCR results and create comprehensive analysis"""
    
    print(f"💾 Saving results to {output_folder}...")
    
    # Create output directory
    os.makedirs(output_folder, exist_ok=True)
    
    # 1. Save complete results as JSON
    results_file = os.path.join(output_folder, "complete_ocr_results.json")
    with open(results_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"✅ Complete results saved to: {results_file}")
    
    # 2. Create summary DataFrame and save as CSV
    summary_data = []
    for result in results:
        summary_data.append({
            'filename': result['filename'],
            'success': result['success'],
            'lines_found': result['lines_found'],
            'lines_with_text': result['lines_with_text'],
            'text_length': len(result['full_text']),
            'processing_time': result['processing_time'],
            'first_50_chars': result['full_text'][:50] + '...' if len(result['full_text']) > 50 else result['full_text']
        })
    
    summary_df = pd.DataFrame(summary_data)
    summary_file = os.path.join(output_folder, "ocr_summary.csv")
    summary_df.to_csv(summary_file, index=False, encoding='utf-8')
    print(f"✅ Summary CSV saved to: {summary_file}")
    
    # 3. Save extracted text as plain text files
    text_folder = os.path.join(output_folder, "extracted_texts")
    os.makedirs(text_folder, exist_ok=True)
    
    for result in results:
        if result['success'] and result['full_text']:
            text_filename = os.path.splitext(result['filename'])[0] + ".txt"
            text_file = os.path.join(text_folder, text_filename)
            
            with open(text_file, 'w', encoding='utf-8') as f:
                f.write(f"Source Image: {result['filename']}\\n")
                f.write(f"Processing Time: {result['processing_time']:.2f}s\\n")
                f.write(f"Lines Found: {result['lines_found']}\\n")
                f.write(f"Lines with Text: {result['lines_with_text']}\\n")
                f.write("-" * 50 + "\\n")
                f.write(result['full_text'])
    
    print(f"✅ Individual text files saved to: {text_folder}")
    
    # 4. Create analysis report
    analysis = analyze_ocr_results(results)
    analysis_file = os.path.join(output_folder, "analysis_report.json")
    with open(analysis_file, 'w', encoding='utf-8') as f:
        json.dump(analysis, f, ensure_ascii=False, indent=2)
    print(f"✅ Analysis report saved to: {analysis_file}")
    
    return summary_df, analysis

def analyze_ocr_results(results: List[dict]) -> dict:
    """Create comprehensive analysis of OCR results"""
    
    successful_results = [r for r in results if r['success']]
    failed_results = [r for r in results if not r['success']]
    
    analysis = {
        'overall_statistics': {
            'total_images': len(results),
            'successful_images': len(successful_results),
            'failed_images': len(failed_results),
            'success_rate': (len(successful_results) / len(results) * 100) if results else 0
        },
        'processing_times': {
            'average_time': np.mean([r['processing_time'] for r in results]),
            'min_time': np.min([r['processing_time'] for r in results]),
            'max_time': np.max([r['processing_time'] for r in results])
        },
        'text_statistics': {
            'total_lines_found': sum(r['lines_found'] for r in successful_results),
            'total_lines_with_text': sum(r['lines_with_text'] for r in successful_results),
            'total_characters_extracted': sum(len(r['full_text']) for r in successful_results),
            'average_text_per_image': np.mean([len(r['full_text']) for r in successful_results]) if successful_results else 0
        },
        'quality_metrics': {
            'line_recognition_rate': 0,
            'average_lines_per_image': np.mean([r['lines_found'] for r in successful_results]) if successful_results else 0
        }
    }
    
    # Calculate line recognition rate
    if analysis['text_statistics']['total_lines_found'] > 0:
        analysis['quality_metrics']['line_recognition_rate'] = (
            analysis['text_statistics']['total_lines_with_text'] / 
            analysis['text_statistics']['total_lines_found'] * 100
        )
    
    return analysis

def create_visualization(summary_df: pd.DataFrame, analysis: dict):
    """Create visualizations of OCR results"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Success Rate Pie Chart
    success_counts = summary_df['success'].value_counts()
    axes[0, 0].pie(success_counts.values, labels=['Success', 'Failed'], autopct='%1.1f%%', startangle=90)
    axes[0, 0].set_title('OCR Success Rate')
    
    # 2. Processing Time Distribution
    axes[0, 1].hist(summary_df['processing_time'], bins=10, alpha=0.7, color='skyblue')
    axes[0, 1].set_title('Processing Time Distribution')
    axes[0, 1].set_xlabel('Time (seconds)')
    axes[0, 1].set_ylabel('Frequency')
    
    # 3. Lines Found vs Lines with Text
    successful_data = summary_df[summary_df['success'] == True]
    if len(successful_data) > 0:
        axes[1, 0].scatter(successful_data['lines_found'], successful_data['lines_with_text'], alpha=0.6)
        axes[1, 0].plot([0, successful_data['lines_found'].max()], [0, successful_data['lines_found'].max()], 'r--', alpha=0.5)
        axes[1, 0].set_title('Lines Found vs Lines with Text')
        axes[1, 0].set_xlabel('Lines Found')
        axes[1, 0].set_ylabel('Lines with Text')
    
    # 4. Text Length Distribution
    if len(successful_data) > 0:
        axes[1, 1].hist(successful_data['text_length'], bins=10, alpha=0.7, color='lightgreen')
        axes[1, 1].set_title('Extracted Text Length Distribution')
        axes[1, 1].set_xlabel('Text Length (characters)')
        axes[1, 1].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\\n" + "="*50)
    print("📊 OCR ANALYSIS SUMMARY")
    print("="*50)
    print(f"📈 Overall Statistics:")
    print(f"   Total Images: {analysis['overall_statistics']['total_images']}")
    print(f"   Success Rate: {analysis['overall_statistics']['success_rate']:.1f}%")
    print(f"   Average Processing Time: {analysis['processing_times']['average_time']:.2f}s")
    
    print(f"\\n📝 Text Extraction:")
    print(f"   Total Lines Found: {analysis['text_statistics']['total_lines_found']}")
    print(f"   Total Lines with Text: {analysis['text_statistics']['total_lines_with_text']}")
    print(f"   Line Recognition Rate: {analysis['quality_metrics']['line_recognition_rate']:.1f}%")
    print(f"   Total Characters: {analysis['text_statistics']['total_characters_extracted']}")
    print(f"   Avg Characters per Image: {analysis['text_statistics']['average_text_per_image']:.1f}")

# If we have test results, save and analyze them
if 'test_results' in locals() and test_results:
    print("💾 Saving and analyzing test results...")
    
    # Save results
    summary_df, analysis = save_and_analyze_results(test_results, OUTPUT_FOLDER)
    
    # Create visualizations
    create_visualization(summary_df, analysis)
    
    print(f"\\n✅ All results saved to: {OUTPUT_FOLDER}")
    print("\\n📁 Files created:")
    print("   - complete_ocr_results.json (detailed results)")
    print("   - ocr_summary.csv (summary table)")
    print("   - extracted_texts/ (individual text files)")
    print("   - analysis_report.json (comprehensive analysis)")
else:
    print("⚠️  No test results available to save. Run the OCR pipeline first.")

## 12. Process Complete Dataset
Run the complete OCR pipeline on all images in the dataset.

In [ ]:
# WARNING: This will process ALL images in your dataset
# Uncomment and run only when you're ready for full processing

PROCESS_FULL_DATASET = False  # Set to True to process all images

if PROCESS_FULL_DATASET and len(clean_dataset_df) > 0:
    print("🚀 STARTING FULL DATASET PROCESSING!")
    print(f"📊 Total images to process: {len(clean_dataset_df)}")
    print("⏰ This may take a considerable amount of time...")
    
    # Confirm before proceeding
    import time
    print("⏳ Starting in 10 seconds... (interrupt if needed)")
    time.sleep(10)
    
    # Process the complete dataset
    full_results = ocr_pipeline.process_dataset(clean_dataset_df)
    
    # Save and analyze results
    final_output_folder = os.path.join(OUTPUT_FOLDER, "full_dataset_results")
    summary_df, analysis = save_and_analyze_results(full_results, final_output_folder)
    
    # Create comprehensive visualizations
    create_visualization(summary_df, analysis)
    
    print("\\n" + "="*60)
    print("🎉 FULL DATASET PROCESSING COMPLETE!")
    print("="*60)
    print(f"📁 All results saved to: {final_output_folder}")
    
    # Final statistics
    final_stats = ocr_pipeline.get_pipeline_stats()
    print(f"\\n📈 Final Pipeline Statistics:")
    for key, value in final_stats.items():
        print(f"   {key.replace('_', ' ').title()}: {value}")
        
else:
    print("\\n📋 To process the full dataset:")
    print("   1. Set PROCESS_FULL_DATASET = True")
    print("   2. Run this cell")
    print("   3. Wait for completion")
    print(f"\\n📊 Dataset ready for processing: {len(clean_dataset_df)} valid images")
    print(f"💾 Results will be saved to: {OUTPUT_FOLDER}")

print(\"\\n\" + \"=\"*60)
print(\"🎯 SANSKRIT OCR PIPELINE READY!\")
print(\"=\"*60)
print(\"✅ All components initialized and tested\")
print(\"📚 Dataset loaded and validated\")
print(\"🔧 Preprocessing pipeline ready\")
print(\"🔪 Line segmentation working\")
print(\"🤖 AI text recognition available\")
print(\"💾 Results saving system ready\")
print(\"\\nYour Sanskrit OCR system is fully operational!\")